# Phase 4: Deep Learning (The Uber DeepETA Approach)
Neural networks traditionally struggle with tabular data. If we just feed raw numbers into a standard PyTorch Linear layer, XGBoost will beat it easily.

**Uber's Solution in DeepETA:** They took categorical features (like `segment_id`, `day_of_week`) and passed them through **Embedding Layers**. This allows the neural network to learn a dense vector representation of each bus stop, grouping similar stops together in a mathematical space. 

In this notebook, we will build a PyTorch model that does exactly this!

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 1. Load Data
df = pd.read_csv('../data/processed/engineered_running_times.csv')

# 2. Preprocessing for PyTorch
le_segment = LabelEncoder()
df['segment_idx'] = le_segment.fit_transform(df['segment'])

categorical_cols = ['segment_idx', 'day_of_week', 'direction']
continuous_cols = ['length', 'hour_sin', 'hour_cos', 'avg_segment_time_at_hour', 'is_weekend', 'is_rush_hour']

# Scale continuous features
scaler = StandardScaler()
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

# Split Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# 3. Create PyTorch Dataset
class BusETADataset(Dataset):
    def __init__(self, dataframe, cat_cols, cont_cols, target_col):
        self.cat_X = dataframe[cat_cols].values.astype(np.int64)
        self.cont_X = dataframe[cont_cols].values.astype(np.float32)
        self.y = dataframe[target_col].values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat_X[idx], self.cont_X[idx], self.y[idx]

train_dataset = BusETADataset(train_df, categorical_cols, continuous_cols, 'run_time_in_seconds')
test_dataset = BusETADataset(test_df, categorical_cols, continuous_cols, 'run_time_in_seconds')

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# 4. Define the DeepETA-style Architecture
class ETANeuralNet(nn.Module):
    def __init__(self, embedding_sizes, num_continuous):
        super(ETANeuralNet, self).__init__()
        
        self.embeddings = nn.ModuleList([nn.Embedding(categories, size) for categories, size in embedding_sizes])
        
        num_embeddings = sum(size for _, size in embedding_sizes)
        input_dim = num_embeddings + num_continuous
        
        self.fc_layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_cat, x_cont):
        embedded = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
        x_cat_embedded = torch.cat(embedded, dim=1)
        
        x = torch.cat([x_cat_embedded, x_cont], dim=1)
        return self.fc_layers(x).squeeze()

embedding_sizes = [
    (df['segment_idx'].nunique(), 10),
    (7, 4),
    (3, 2)
]

model = ETANeuralNet(embedding_sizes, num_continuous=len(continuous_cols))

In [ ]:
# 5. Training Loop
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 15
model.train()

for epoch in range(epochs):
    total_loss = 0
    for x_cat, x_cont, y in train_loader:
        optimizer.zero_grad()
        predictions = model(x_cat, x_cont)
        loss = criterion(predictions, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Training Loss (MSE): {total_loss/len(train_loader):.2f}")

Epoch 1/15 - Training Loss (MSE): 5694.08
Epoch 2/15 - Training Loss (MSE): 4127.95
Epoch 3/15 - Training Loss (MSE): 4100.32
Epoch 4/15 - Training Loss (MSE): 4069.63
Epoch 5/15 - Training Loss (MSE): 4054.97
Epoch 6/15 - Training Loss (MSE): 4041.21
Epoch 7/15 - Training Loss (MSE): 4033.76
Epoch 8/15 - Training Loss (MSE): 4013.89
Epoch 9/15 - Training Loss (MSE): 4006.86
Epoch 10/15 - Training Loss (MSE): 3992.57
Epoch 11/15 - Training Loss (MSE): 3999.46
Epoch 12/15 - Training Loss (MSE): 3975.81
Epoch 13/15 - Training Loss (MSE): 3972.77
Epoch 14/15 - Training Loss (MSE): 3989.19
Epoch 15/15 - Training Loss (MSE): 3963.10


In [ ]:
# 6. Evaluation (Updated to match Classical ML Metrics)
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x_cat, x_cont, y in test_loader:
        predictions = model(x_cat, x_cont)
        all_preds.extend(predictions.numpy())
        all_targets.extend(y.numpy())

print("\n--- DeepETA (PyTorch MLP) Results ---")
print(f"MAE:  {mean_absolute_error(all_targets, all_preds):.2f} seconds")
print(f"RMSE: {np.sqrt(mean_squared_error(all_targets, all_preds)):.2f} seconds")
print(f"R2:   {r2_score(all_targets, all_preds):.4f}\n")


--- DeepETA (PyTorch MLP) Results ---
MAE:  36.06 seconds
RMSE: 61.14 seconds
R2:   0.7588

